# Querying Using the 3-Level Namespace


Explicitly addressing table using 3-level namespace


In [0]:
%sql
SELECT 
    pickup_zip,
    COUNT(1) AS total_trips,
    ROUND(AVG(trip_distance), 2) AS avg_distance,
    ROUND(AVG(fare_amount), 2) AS avg_fare
FROM samples.nyctaxi.trips
WHERE trip_distance > 0 AND fare_amount > 0
GROUP BY pickup_zip
ORDER BY total_trips DESC
LIMIT 10;


Alternative to full 3-level naming


In [0]:
%sql
USE CATALOG samples;

USE SCHEMA nyctaxi;

-- Now the table can be queried directly with 1-level name:
SELECT COUNT(*) FROM trips;


# Unity Catalog information_schema

Databricks Unity Catalog Information Schema Query


In [0]:
%sql
SELECT 
    table_catalog,
    table_schema,
    table_name,
    table_type,
    created_by
FROM samples.information_schema.tables
WHERE table_schema = 'nyctaxi'
ORDER BY table_name;


In [0]:
%sql
SELECT 
    column_name,
    data_type,
    is_nullable,
    ordinal_position
FROM samples.information_schema.columns
WHERE table_catalog = 'samples'
  AND table_schema = 'nyctaxi'
  AND table_name = 'trips'
ORDER BY ordinal_position;


# Granular Access Control Lab (GRANT, Test, REVOKE)

In [0]:
%sql
-- Step 1: Grant hierarchy to lab group
GRANT USE CATALOG ON CATALOG hpe_catalog TO `data_analysts`;
GRANT USE SCHEMA ON SCHEMA hpe_catalog.nyc TO `data_analysts`;
GRANT SELECT ON TABLE hpe_catalog.nyc.taxi_zone_lookup TO `data_analysts`;

-- Step 2: Have a partner belonging to `data_analysts` test access
SELECT * FROM hpe_catalog.nyc.taxi_zone_lookup LIMIT 10;
-- Result: SUCCESS

-- Step 3: Revoke an intermediate requirement
REVOKE USE SCHEMA ON SCHEMA hpe_catalog.nyc FROM `data_analysts`;

-- Step 4: Have partner re-run the exact same query
SELECT * FROM hpe_catalog.nyc.taxi_zone_lookup LIMIT 10;
-- Result: PERMISSION DENIED (Cannot traverse down the hierarchy despite SELECT on the table remaining intact)

-- Step 5: Restore correct permissions
GRANT USE SCHEMA ON SCHEMA hpe_catalog.nyc TO `data_analysts`;
